In [ ]:
!pip install -q faiss-cpu sentence-transformers rank_bm25 pypdf pdfplumber huggingface_hub transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 996.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 118.4 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

files_list = list(uploaded.keys())
print("Uploaded files:", files_list)

Saving 4-top-sureshot-pattern-in-quotex-binary-trading_compress.pdf to 4-top-sureshot-pattern-in-quotex-binary-trading_compress.pdf
Saving 18-sureshot-pattern_compress.pdf to 18-sureshot-pattern_compress.pdf
Saving 30-sure-shot-pattern-binary-option_compress.pdf to 30-sure-shot-pattern-binary-option_compress.pdf
Saving 60-second-binary-options-strategy-the-complete-guide-pdf_compress.pdf to 60-second-binary-options-strategy-the-complete-guide-pdf_compress.pdf
Saving 60-seconds-binary-options-secret-proven-strategy-how-to-pdf_compress.pdf to 60-seconds-binary-options-secret-proven-strategy-how-to-pdf_compress.pdf
Saving 60-seconds-binary-options-strategy_compress.pdf to 60-seconds-binary-options-strategy_compress.pdf
Saving 67f95575431873663b8b4592_compress (1).pdf to 67f95575431873663b8b4592_compress (1).pdf
Saving 67f95575431873663b8b4592_compress.pdf to 67f95575431873663b8b4592_compress.pdf
Saving 4194-quotex-bot-trading-binary-bot-signls-99-working-youtube_compress (1).pdf to 4194-q

In [7]:
import os
import re
import faiss
import numpy as np
from pypdf import PdfReader
import pdfplumber
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from huggingface_hub import InferenceClient

# =========================
# CONFIG
# =========================
HF_TOKEN = ""  # 🔴 put your token here

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

client = InferenceClient(
    model="meta-llama/Meta-Llama-3-8B-Instruct",
    token=HF_TOKEN
)

chat_history = []

# =========================
# CLEAN TEXT
# =========================
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# =========================
# TOKENIZE
# =========================
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    return text.split()

# =========================
# LOAD PDF
# =========================
def load_pdfs(files):
    docs = []

    for file in files:
        try:
            reader = PdfReader(file)
            pages = [p.extract_text() for p in reader.pages]
        except:
            pages = []
            with pdfplumber.open(file) as pdf:
                for page in pdf.pages:
                    pages.append(page.extract_text())

        for i, text in enumerate(pages):
            if not text:
                continue

            docs.append({
                "text": clean_text(text),
                "source": file,
                "page": i
            })

    return docs

# =========================
# CHUNKING
# =========================
def chunk_text(docs, chunk_size=400, overlap=50):
    chunks = []

    for doc in docs:
        words = doc["text"].split()

        for i in range(0, len(words), chunk_size - overlap):
            chunk = " ".join(words[i:i + chunk_size])

            chunks.append({
                "text": chunk,
                "source": doc["source"],
                "page": doc["page"]
            })

    return chunks

# =========================
# FAISS
# =========================
def build_faiss(chunks):
    texts = [c["text"] for c in chunks]

    embeddings = embed_model.encode(
        texts,
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    embeddings = np.array(embeddings).astype("float32")
    dim = embeddings.shape[1]

    index = faiss.IndexIDMap(faiss.IndexFlatIP(dim))
    ids = np.arange(len(chunks))

    index.add_with_ids(embeddings, ids)

    return index

# =========================
# BM25
# =========================
def build_bm25(chunks):
    tokenized = [tokenize(c["text"]) for c in chunks]
    return BM25Okapi(tokenized)

# =========================
# HYBRID SEARCH
# =========================
def hybrid_search(query, faiss_index, bm25, chunks, k=8, alpha=0.7):
    q_vec = embed_model.encode([query], normalize_embeddings=True)
    q_vec = np.array(q_vec).astype("float32")

    D, I = faiss_index.search(q_vec, k * 2)
    dense_scores = {idx: score for idx, score in zip(I[0], D[0])}

    sparse_scores = bm25.get_scores(tokenize(query))
    sparse_scores = sparse_scores / (np.max(sparse_scores) + 1e-6)

    combined = {}

    for i in range(len(chunks)):
        d = dense_scores.get(i, 0)
        s = sparse_scores[i]
        combined[i] = alpha * d + (1 - alpha) * s

    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)
    top_idx = [i for i, _ in ranked[:k]]

    return [chunks[i] for i in top_idx]

# =========================
# RERANK
# =========================
def rerank(query, results, top_k=3):
    if not results:
        return []

    pairs = [(query, r["text"]) for r in results]
    scores = reranker.predict(pairs, batch_size=16)

    ranked = sorted(zip(results, scores), key=lambda x: x[1], reverse=True)

    final = []
    for r, s in ranked[:top_k]:
        r_copy = r.copy()
        r_copy["score"] = float(s)
        final.append(r_copy)

    return final

# =========================
# PROMPT
# =========================
def build_prompt(query, context):
    history = "\n".join([
        f"{m['role'].capitalize()}: {m['content']}"
        for m in chat_history[-6:]
    ])

    return f"""
You are a trading assistant.

RULES:
- Use ONLY context
- If not found → say "Not found in context"

Chat History:
{history}

Context:
{context}

Question:
{query}

Answer:
"""

# =========================
# LLM
# =========================
def ask_llm(query, results):
    if not results:
        return "No relevant context found."

    context = "\n\n".join([
        f"[{r['source']} | p{r['page']}]\n{r['text']}"
        for r in results
    ])

    prompt = build_prompt(query, context)

    try:
        response = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200,
            temperature=0.2
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {str(e)}"

# =========================
# BUILD SYSTEM
# =========================
print("🔄 Loading PDFs...")
docs = load_pdfs(files_list)

print("✂️ Chunking...")
chunks = chunk_text(docs)

print("⚡ Building FAISS...")
faiss_index = build_faiss(chunks)

print("📚 Building BM25...")
bm25 = build_bm25(chunks)

print("✅ SYSTEM READY")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Loading PDFs...
✂️ Chunking...
⚡ Building FAISS...


Batches:   0%|          | 0/73 [00:00<?, ?it/s]

📚 Building BM25...
✅ SYSTEM READY


In [8]:
def ask(query):
    results = hybrid_search(query, faiss_index, bm25, chunks)
    results = rerank(query, results)

    answer = ask_llm(query, results)

    chat_history.append({"role": "user", "content": query})
    chat_history.append({"role": "assistant", "content": answer})

    print("\n🤖", answer)

In [12]:
ask("what kind strategy help to profit?")



🤖 Based on the context, it seems that the "BUG strategy" is mentioned as a strategy that can help earn a good profit in binary trading with high accuracy (95%-100%). 

However, the context also mentions that the success of the strategy depends on the trader's ability to accurately determine the current state of the market and whether it is ranging or trending. 

Additionally, the context suggests that a robot-based strategy, such as the HBSwiss System, can also help maximize gains and minimize the risk of financial losses.
